## Запуск подключения и вытягивания данных из БД

In [1]:
import sys
sys.path.insert(1, r'D:\Khabarov\Репозиторий\sql_premises_and_volumes')
from Helpers.ComplexCOHelper import ComplexCOHelper

common_source_path = r'D:\Khabarov\Репозиторий\sql_premises_and_volumes\SourceData\ИсходныеДанные_ССК_Башни.xlsx'
compHel = ComplexCOHelper(volume_source_path=common_source_path,premise_source_path=common_source_path)

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
Подключено


D:\Khabarov\Репозиторий\sql_premises_and_volumes\Helpers\DbConnector.py:525: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dfFull.groupby('name', group_keys=False).apply(lambda x: x.nlargest(1, "version_index"))[
D:\Khabarov\Репозиторий\sql_premises_and_volumes\Helpers\DbConnector.py:275: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  latest_calcs = df_calc_ids.groupby('model_version_id', group_keys=False).apply(lambd

Завершен сбор id по необходимым версиям моделей
Завершен сбор параметров распознавания
Завершен сбор параметров стандартизации
Завершен сбор параметров расчета
Завершен сбор параметров расположения
Завершено объединение данных
Завершено добавление информации по секциям, этажам и стадиям

Затраченное время: 1645.614113330841, вес: 129.84 МБайт
Подключено
Добавлен ЮКВ08 Стадия П
Добавлен ШГР17.1 Стадия П
Добавлен ТРА01 Стадия П
Добавлен НВЕ01 Стадия П
Добавлен НВЕ02 Стадия П
Добавлен НВЕ03 Стадия П
Добавлен ИЗД01 Стадия П
Добавлен ДМА01.1 Стадия П
Добавлен МОН01 Стадия П
Не удалось загрузить МОН-СИ01/ШК Стадия П
Добавлен МТК05 Стадия П
Не удалось загрузить МТК09/ДС Стадия П
Добавлен МТК08/ПГ Стадия П
Добавлен МТК02 Стадия П
Добавлен МТК01 Стадия П
Добавлен МТК03 Стадия П
Добавлен МТК04 Стадия П
Добавлен МТК07 Стадия П
Добавлен МТК06 Стадия П
Добавлен ПШЦ06.1 Стадия П
Добавлен ЛБД01.1 Стадия П
Добавлен ДВТ06 Стадия П
Добавлен ДВТ09/ПГ Стадия П
Добавлен ДВТ04 Стадия П
Добавлен СВЕ03.1 Стад

## Выгрузка для анализа этажей

In [2]:
# df = compHel.get_floor_types_features_dataset()
# directory = r'D:\Khabarov\Репозиторий\sql_premises_and_volumes\Data\ТипыЭтажей_ДляАнализа_v2.xlsx'
# df.to_excel(directory,sheet_name='Лист1',index=False)
# print('Сохранено')

#Датасет для обучения определения типа этажа

## Выгрузка по ССК башен

### Импорты и функции

In [3]:
import numpy as np
import pandas as pd
from Helpers.ParamsAndFuns import ParamsAndFuns as p
import re

def extract_floor_number(floor_str):
    if pd.isna(floor_str): return None
    # Ищем число в строке, включая знак минус
    match = re.search(r'-?\d+', str(floor_str))
    return int(match.group()) if match else None

### Получаем бетон

In [4]:
df_sk = compHel.df_volumes.copy()

b_sk_names = [
    'Монолитный пилон',
'Монолитная стена',
'Монолитный марш',
'Монолитная плита',
'Монолитная площадка',
'Сборный марш',
'Сборная площадка',
'Сборный балкон',
'Префаб. Вертикальная конструкция',
'Префаб. Горизонтальная конструкция',
'Сборный отлив',
'Сборная стена подвала',
]
df_b = df_sk[df_sk['Имя СК'].isin(b_sk_names)]
df_b = df_sk

df_sect_b = df_b.groupby(['construction_object_id','Наименование ОС','Секция'],as_index=False).agg(
    V_b=('Объем, м3','sum'),
    Морфотип=('Морфотип секции','first'),
)

### Получаем фасад

In [5]:
fac_sk_names = [
    'Штукатурка фасадная',
    'Навесной фасад',
    'Навесной фасад. Кладка',
    'Навесной фасад. Префаб',
    'Балконный блок',
    'Оконный блок',
    'Витраж'
]
df_fac = df_sk[df_sk['Имя СК'].isin(fac_sk_names)]
df_sect_fac = df_fac.groupby(['Наименование ОС','Секция'],as_index=False).agg(
    S_f=('Площадь изделия','sum'),
)
df_sect_fac

,Наименование ОС,Секция,S_f
0,ДВТ04,Паркинг,0.00000
1,ДВТ04,Секция 1,927.89015
2,ДВТ04,Секция 2,347.22880
3,ДВТ04,Секция 3,265.94515
4,ДВТ04,Секция 4,566.90155
...,...,...,...
193,ЮКВ08,Секция 4,383.70510
194,ЮКВ08,Секция 5,497.04320
195,ЮКВ08,Секция 6,340.04440
196,ЮКВ08,Секция 7,514.43050


### Получаем справочник типов этажей

In [6]:
df_floor_dict = df_b.groupby(['construction_object_id','Наименование ОС','Секция','Этаж'],as_index=False).agg(
    Тип_этажа=('Тип этажа','first'),
)
df_floor_dict['Этаж_число'] = df_floor_dict['Этаж'].apply(extract_floor_number).astype(float)
df_floor_dict = df_floor_dict.drop('Этаж',axis=1)
df_floor_dict = df_floor_dict.rename(mapper={'Этаж_число':'Этаж', 'Секция':'Номер секции'},axis=1)
df_floor_dict = df_floor_dict[['Наименование ОС','Номер секции','Этаж','Тип_этажа']]
df_floor_dict

,Наименование ОС,Номер секции,Этаж,Тип_этажа
0,ПШЦ06.1,Паркинг,-1.0,-1 этаж Паркинг
1,ПШЦ06.1,Секция 1,-1.0,-1 этаж
2,ПШЦ06.1,Секция 1,1.0,1 этаж
3,ПШЦ06.1,Секция 1,2.0,Типовой этаж
4,ПШЦ06.1,Секция 1,3.0,Типовой этаж
...,...,...,...,...
2693,МНС02,"Секция 9,10",5.0,Не заполнено
2694,МНС02,"Секция 9,10",6.0,Не заполнено
2695,МНС02,"Секция 9,10",7.0,Не заполнено
2696,МНС02,"Секция 9,10",8.0,Не заполнено


### Получаем помещения

In [ ]:
#Собираем данные по помещениям
df_premises = compHel.df_premises.copy()
df_premises = df_premises[df_premises['Назначение'].isin(['Окно','Дверь','Витраж','ГНС']) == False]
df_premises = df_premises.dropna(subset=['Назначение'])

# #Получение SFA
# df_premises['is_sfa'] = np.where(
#     (~df_premises['Вид помещения'].isin(['МОП', 'Технические помещения'])),
#     True,
#     False
# )
# df_premises['sfa'] = np.where(df_premises['is_sfa'], df_premises['Площадь помещения'],0)

#Получение SFA жилье
df_premises['is_sfa_liv'] = np.where(
    (~df_premises['Номер секции'].str.contains('Паркинг', na=False)) & 
    (~df_premises['Вид помещения'].isin(['МОП', 'Технические помещения'])),
    True,
    False
)
df_premises['sfa_liv'] = np.where(df_premises['is_sfa_liv'], df_premises['Площадь помещения'],0)

#Получение SFA паркинга
df_premises['is_sfa_park'] = np.where(
    (df_premises['Номер секции'].str.contains('Паркинг', na=False)) & 
    (~df_premises['Вид помещения'].isin(['МОП', 'Технические помещения'])) &
    ((df_premises['Вид помещения'] == 'Машино-место') | (df_premises['Назначение'] == 'Кладовки')),
    True,
    False
)
df_premises['sfa_park'] = np.where(df_premises['is_sfa_park'], df_premises['Площадь помещения'],0)

#Получение GFA
df_premises['is_gfa'] = np.where(
    (~df_premises['Вид помещения'].isin(['Машино-место'])),
    True,
    False
)
df_premises['gfa'] = np.where(df_premises['is_gfa'], df_premises['Площадь помещения'],0)

#Посчет квартир
df_premises['is_flat'] = np.where((df_premises['Назначение'] == 'Жилье') & (df_premises['Вид помещения'] != 'МОП'),
                                  True,
                                  False)
df_premises['flat_num'] = np.where(df_premises['is_flat'],df_premises[p.adsk_premise_number],np.nan)
df_premises['flat_part_area'] = np.where(df_premises['is_flat'],df_premises[p.bru_premise_part_area_pn],0)

#Присобачили тип этажа
df_premises['Этаж'] = df_premises['Этаж'].fillna(0)
df_premises = pd.merge(left=df_premises,right=df_floor_dict,how='left',on=['Наименование ОС','Номер секции','Этаж'])
df_premises['is_typical'] = np.where(df_premises['Тип_этажа'] == 'Типовой этаж',True,False)
df_premises['sfa_typical'] = np.where(df_premises['is_typical'] == True,df_premises['sfa_liv'],0)
df_premises['gfa_typical'] = np.where(df_premises['is_typical'] == True,df_premises[p.bru_premise_part_area_pn],0)
df_premises


,BRU_Количество комнат для продаж,BRU_Тип квартиры,Азимут,Антресоль,Балкон,Без доступа света,Без летнего помещения,Вид во двор,Вид на природный объект,Вид на улицу,...,sfa_park,is_gfa,gfa,is_flat,flat_num,flat_part_area,Тип_этажа,is_typical,sfa_typical,gfa_typical
0,1.0,1С,NaN,0.0,1.0,NaN,0.0,1.0,0.0,0.0,...,0.0,True,2.90,True,4.6.1,2.90,Типовой этаж,True,2.90,2.90
1,3.0,3С,NaN,0.0,1.0,NaN,0.0,1.0,0.0,1.0,...,0.0,True,3.10,True,5.4.1,3.10,Типовой этаж,True,3.10,3.10
2,1.0,1С,NaN,0.0,1.0,NaN,0.0,1.0,0.0,0.0,...,0.0,True,2.80,True,3.4.10,2.80,Типовой этаж,True,2.80,2.80
3,1.0,1С,NaN,0.0,1.0,NaN,0.0,0.0,0.0,1.0,...,0.0,True,4.20,True,4.3.3,4.20,Типовой этаж,True,4.20,4.20
4,2.0,2С,NaN,0.0,1.0,NaN,0.0,1.0,0.0,1.0,...,0.0,True,12.80,True,7.6.1,12.80,Типовой этаж,True,12.80,12.80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98007,2.0,2С,NaN,0.0,0.0,NaN,0.0,1.0,0.0,0.0,...,0.0,True,11.63,True,2.2.1,11.63,Типовой этаж,True,11.63,11.63
98008,3.0,3С,NaN,0.0,0.0,NaN,0.0,0.0,0.0,1.0,...,0.0,True,3.59,True,10.6.4,3.59,Не заполнено,False,0.00,0.00
98009,2.0,2С,NaN,0.0,1.0,NaN,0.0,1.0,0.0,1.0,...,0.0,True,20.18,True,5.5.7,20.18,Не заполнено,False,0.00,0.00
98010,NaN,(Кладовая),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,True,3.81,False,NaN,0.00,Не заполнено,False,0.00,0.00


In [15]:
df_premises[p.bru_floor_int_pn] = pd.to_numeric(df_premises[p.bru_floor_int_pn], errors='coerce')
df_premises[p.bru_floor_int_pn] = df_premises[p.bru_floor_int_pn].fillna(0)

df_sect_areas = df_premises.groupby(['Наименование ОС','Номер секции'],as_index=False).agg(
    AREA=('Площадь помещения','sum'),
    SFA_liv=('sfa_liv','sum'),
    SFA_park=('sfa_park','sum'),
    GFA=('gfa','sum'),
    max_fl=(p.bru_floor_int_pn,'max'),
    N_kv=('flat_num', 'nunique'),
    SFA_flats=('flat_part_area','sum'),
    SFA_typical=('sfa_typical','sum'),
    GFA_typical=('gfa_typical','sum'),
)

# df_premises[['Категория','Назначение','Вид помещения','is_sfa']].value_counts()
df_sect_areas = df_sect_areas.rename(mapper={'Номер секции':'Секция'},axis=1)
df_sect_areas

,Наименование ОС,Секция,AREA,SFA_liv,SFA_park,GFA,max_fl,N_kv,SFA_flats,SFA_typical,GFA_typical
0,ДВТ04,Паркинг,7803.85,0.00,2857.59,4946.26,1.0,0,0.00,0.00,0.00
1,ДВТ04,Секция 1,6527.48,5202.02,0.00,6527.48,15.0,71,4885.82,4885.82,5572.84
2,ДВТ04,Секция 2,2940.72,2219.86,0.00,2940.72,8.0,38,1916.94,1801.91,2095.21
3,ДВТ04,Секция 3,2079.65,1496.20,0.00,2079.65,8.0,22,1379.04,1292.69,1551.20
4,ДВТ04,Секция 4,4826.75,3812.52,0.00,4826.75,10.0,69,3316.35,3316.35,3796.31
...,...,...,...,...,...,...,...,...,...,...,...
197,ЮКВ08,Секция 4,3156.10,2362.40,0.00,3156.10,9.0,37,2139.30,1846.20,2114.00
198,ЮКВ08,Секция 5,4074.20,3140.50,0.00,4074.20,10.0,43,2849.50,2849.50,3272.80
199,ЮКВ08,Секция 6,2853.00,2182.90,0.00,2853.00,8.0,29,1840.90,1703.00,1961.20
200,ЮКВ08,Секция 7,4077.70,3148.10,0.00,4077.70,10.0,42,2746.30,2746.30,3133.20


In [17]:
#Суммируем
res = pd.merge(left=df_sect_areas,right=df_sect_b,how='left',on=['Наименование ОС','Секция'])
res = pd.merge(left=res,right=df_sect_fac,how='left',on=['Наименование ОС','Секция'])

res = res[
    # (res['Морфотип'].isin(tower_morphotypes)) &
     (res['max_fl'] > 16)
          ]
res['K1'] = round(res['SFA_typical'] / res['GFA_typical'],2)
res['K2'] = round(res['SFA_liv'] / res['GFA'],2)
res['Sср'] = round(res['SFA_flats'] / res['N_kv'],2)
res['кб'] = round(res['V_b'] / res['SFA_liv'],2)
res['кф'] = round(res['S_f'] / res['SFA_liv'],2)
res

,Наименование ОС,Секция,AREA,SFA_liv,SFA_park,GFA,max_fl,N_kv,SFA_flats,SFA_typical,GFA_typical,construction_object_id,V_b,Морфотип,S_f,K1,K2,Sср,кб,кф
8,ДВТ06,Секция 1,6641.42,5199.98,0.0,6641.42,17.0,97,4895.39,4895.39,5683.64,18408c64-049e-43c3-ae2d-bea715fceab4,2549.122852,C_50TS(TN)5.8_Шаг 3.45,1059.934750,0.86,0.78,50.47,0.49,0.20
21,ДМА01.1,Секция 3,9750.60,7738.90,0.0,9750.60,23.0,99,7363.80,0.00,0.00,53e2dd2e-e50c-11ed-b5ea-0242ac120002,230.463120,Уникальная_Шаг 3.45,750.421800,NaN,0.79,74.38,0.03,0.10
26,ДМА01.1,Секция 8,9731.10,7584.30,0.0,9731.10,23.0,99,7363.10,0.00,0.00,53e2dd2e-e50c-11ed-b5ea-0242ac120002,225.611100,Уникальная_Шаг 3.45,754.262700,NaN,0.78,74.37,0.03,0.10
30,ИЗД01,Секция 3,18554.50,15112.20,0.0,18554.50,30.0,253,13205.10,12357.40,14741.50,51949966-5aa8-11ed-9b6a-0242ac120002,1749.024030,Уникальная_Шаг 3.45,3063.263475,0.84,0.81,52.19,0.12,0.20
31,ИЗД01,Секция 4,15451.90,12112.30,0.0,15451.90,25.0,157,10580.80,9606.00,11801.30,51949966-5aa8-11ed-9b6a-0242ac120002,1230.346651,Уникальная_Шаг 3.45,2350.840050,0.81,0.78,67.39,0.10,0.19
32,ИЗД01,Секция 5,19313.90,15804.70,0.0,19313.90,30.0,215,12825.50,11998.20,14476.50,51949966-5aa8-11ed-9b6a-0242ac120002,1548.891372,Уникальная_Шаг 3.45,3261.359150,0.83,0.82,59.65,0.10,0.21
53,МОН01,Секция 1,16713.90,13429.66,0.0,16713.90,29.0,186,13217.76,12596.67,14663.18,eaa6edb8-58e9-11ed-9b6a-0242ac120002,8580.355210,B_75TS(TN)6.10_Шаг 3.45,0.000000,0.86,0.80,71.06,0.64,0.00
57,МОН01,Секция 2,14786.46,11860.55,0.0,14786.46,25.0,169,11720.45,10925.27,12661.78,eaa6edb8-58e9-11ed-9b6a-0242ac120002,7663.128247,B_75TS(TN)6.10_Шаг 3.45,0.000000,0.86,0.80,69.35,0.65,0.00
58,МОН01,Секция 3,16616.92,13443.41,0.0,16616.92,29.0,180,13264.21,12610.60,14730.14,eaa6edb8-58e9-11ed-9b6a-0242ac120002,8620.818267,B_75TS(TN)6.10_Шаг 3.45,0.000000,0.86,0.81,73.69,0.64,0.00
59,МОН01,Секция 4,16696.64,12910.68,0.0,16696.64,27.0,195,12722.98,11987.26,13955.70,eaa6edb8-58e9-11ed-9b6a-0242ac120002,8365.795929,B_75TS(TN)6.10_Шаг 3.45,0.000000,0.86,0.77,65.25,0.65,0.00


In [18]:
directory = r'D:\Khabarov\Репозиторий\sql_premises_and_volumes\Data'
name = r'\Башни_ССК'
res[['construction_object_id','Наименование ОС','Секция','Морфотип','K1','K2','Sср','кб','кф']].to_excel(f"{directory}{name}.xlsx",index=False,sheet_name='Лист1')